In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Curva de geração de vapor × consumo de gás — caldeiras 208, 502 e 550
# MAGIC - **Eixo X:** consumo de gás (Nm³/h) · **Eixo Y:** geração de vapor (t/h) · base horária
# MAGIC - Só horas **em operação** (gás e vapor acima de 5% do p95), após limpeza: negativos → 0 e lacunas curtas de compressão preenchidas (até 12 h)
# MAGIC - Ajuste **linear** (vapor = a + b·gás) e **quadrático** por caldeira e por ano, com **uma tabela por ano**

# COMMAND ----------

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BRONZE_DIR = "/Workspace/Previsao de consumo de vapor/1_bronze/processo_dados"
RESULT_DIR = "/Workspace/Previsao de consumo de vapor/3_gold"
CALDEIRAS = {'208': ('11FI208-02', '11FC208-106'), '502': ('11FI502-30', '11FC502-01'), '550': ('21FI550-30', '21FC550-72')}  # (vapor t/h, gás Nm³/h)
LIMITE_FFILL_H = 12  # repete o último valor por até 12 h (compressão do historiador)
FRAC_OPERACAO = 0.05  # hora em operação: gás e vapor > 5% do p95 da própria tag
MIN_HORAS_ANO = 100  # mínimo de horas em operação para ajustar a curva do ano
COR_ANO = {2023: '#2a78d6', 2024: '#eb6834', 2025: '#1baf7a', 2026: '#eda100'}  # cor fixa por ano

norm = lambda t: str(t).upper().replace('.PV', '').strip()
arquivos = {}
for root, _, files in os.walk(BRONZE_DIR):
    for f in files:
        if f.endswith('.csv') and not re.search(r'\d{4}-\d{2}-\d{2}', f):
            arquivos.setdefault(norm(f.rsplit('.csv', 1)[0]), []).append(os.path.join(root, f))

def carregar(tag):
    frames = [pd.read_csv(f) for f in sorted(arquivos.get(norm(tag), []))]
    if not frames:
        raise FileNotFoundError(f"Tag {tag} não encontrada no bronze")
    df = pd.concat(frames)
    df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    s = df.dropna(subset=['ts']).drop_duplicates('ts').set_index('ts')['value'].sort_index()
    return s.clip(lower=0).resample('1h').mean().ffill(limit=LIMITE_FFILL_H)

dados = {}
for c, (tag_v, tag_g) in CALDEIRAS.items():
    df = pd.DataFrame({'vapor': carregar(tag_v), 'gas': carregar(tag_g)}).dropna()
    lim_v, lim_g = FRAC_OPERACAO * df['vapor'].quantile(0.95), FRAC_OPERACAO * df['gas'].quantile(0.95)
    op = df[(df['vapor'] > lim_v) & (df['gas'] > lim_g)].copy()
    op['ano'] = op.index.year
    dados[c] = op
    print(f"Caldeira {c}: {len(df):,} h com vapor e gás | {len(op):,} h em operação | anos: {sorted(op['ano'].unique().tolist())}")

# COMMAND ----------

linhas, curvas = [], {}
for c, op in dados.items():
    for ano, g in op.groupby('ano'):
        if len(g) < MIN_HORAS_ANO:
            continue
        x, y = g['gas'].values, g['vapor'].values
        b, a = np.polyfit(x, y, 1)
        q2, q1, q0 = np.polyfit(x, y, 2)
        r2_lin = 1 - ((y - (a + b * x)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        r2_quad = 1 - ((y - (q0 + q1 * x + q2 * x ** 2)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        curvas[(c, ano)] = (a, b, q0, q1, q2, x.min(), x.max())
        linhas.append({'ano': ano, 'caldeira': c, 'horas_operacao': len(g), 'gas_min (Nm3/h)': x.min(), 'gas_max (Nm3/h)': x.max(), 'vapor_medio (t/h)': y.mean(), 'a (t/h)': a, 'b (t/Nm3)': b, 'R2_linear': r2_lin, 'q0': q0, 'q1': q1, 'q2': q2, 'R2_quadratico': r2_quad, 'vapor_por_1000Nm3 (t)': np.median(y / x) * 1000, 'gas_por_t_vapor (Nm3/t)': np.median(x / y)})
tab = pd.DataFrame(linhas).sort_values(['ano', 'caldeira']).reset_index(drop=True)
for ano, t in tab.groupby('ano'):
    print(f"=== {ano} — curva vapor × gás por caldeira ===")
    display(t.drop(columns='ano').round(6))
os.makedirs(RESULT_DIR, exist_ok=True)
tab.to_csv(f"{RESULT_DIR}/curva_gas_vapor_por_ano.csv", index=False)
print(f"Tabela salva em {RESULT_DIR}/curva_gas_vapor_por_ano.csv")

# COMMAND ----------

for c, op in dados.items():
    anos_c = sorted(a for (cc, a) in curvas if cc == c)
    if not anos_c:
        continue
    fig, axes = plt.subplots(1, len(anos_c) + 1, figsize=(5.2 * (len(anos_c) + 1), 4.6), sharex=True, sharey=True)
    for ax, ano in zip(axes, anos_c):
        g = op[op['ano'] == ano].sample(min(4000, int((op['ano'] == ano).sum())), random_state=42)
        a, b, q0, q1, q2, xmin, xmax = curvas[(c, ano)]
        xx = np.linspace(xmin, xmax, 100)
        ax.scatter(g['gas'], g['vapor'], s=6, alpha=0.25, color=COR_ANO.get(ano, '#2a78d6'), linewidths=0)
        ax.plot(xx, a + b * xx, color='#222222', lw=2, label='linear')
        ax.plot(xx, q0 + q1 * xx + q2 * xx ** 2, color='#222222', lw=2, ls='--', label='quadrática')
        r2 = tab.loc[(tab['caldeira'] == c) & (tab['ano'] == ano), 'R2_linear'].iloc[0]
        ax.set_title(f"{c} — {ano}  (R² linear = {r2:.3f})", fontsize=10)
        ax.set_xlabel('Consumo de gás (Nm³/h)')
        ax.grid(alpha=0.3)
    axes[0].set_ylabel('Geração de vapor (t/h)')
    axes[0].legend(fontsize=8, loc='upper left')
    for ano in anos_c:
        a, b, q0, q1, q2, xmin, xmax = curvas[(c, ano)]
        xx = np.linspace(xmin, xmax, 100)
        axes[-1].plot(xx, q0 + q1 * xx + q2 * xx ** 2, lw=2, color=COR_ANO.get(ano, '#2a78d6'), label=str(ano))
    axes[-1].set_title(f'{c} — curvas por ano (quadrática)', fontsize=10)
    axes[-1].set_xlabel('Consumo de gás (Nm³/h)')
    axes[-1].legend(fontsize=8, loc='upper left')
    axes[-1].grid(alpha=0.3)
    fig.suptitle(f'Caldeira {c}: geração de vapor × consumo de gás', fontweight='bold')
    plt.tight_layout()
    plt.show()
